# ZestXML: GPU benchmark + the experiments a CPU sandbox could not run

Everything here exists because it was **blocked** where the port was written: no GPU, and
HuggingFace unreachable, so no pretrained transformer of any kind.

**Runtime → Change runtime type → A100 GPU.**

Stage 0 sets up everything; stages 1-4 and 6 assume it has run. **Stage 5 (Renee) is
fully self-contained** — it clones and builds what it needs, so it can be run first, or
alone on a fresh runtime. Rough costs on an A100:

| stage | what it settles | ~time |
| --- | --- | --- |
| 0 setup + data | — | 5 min |
| 1 GPU benchmark, GZ-Eurlex-4.3K | is the port actually fast on GPU, and does it hold at 4.3k labels? | 25–40 min |
| 2 fixed SPLADE | was the zero-shot collapse a leak or the architecture? | 10 min |
| 3 pretrained encoder probe | **the decisive one** — does dense retrieval beat the 71.8% ceiling? | 15 min |
| 4 hybrid shortlist end-to-end | does extra recall convert into accuracy? | 15 min |
| 5 Renee (Microsoft, MLSys'23) | how far is the port from a current XMC method? | 45–90 min |
| 6 comparison table | — | 1 min |

Stage 3 is the highest information per minute. If you only run one, run that one.


## Stage 0 — setup and data

In [ ]:
!nvidia-smi
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
%cd /content
!rm -rf zestxml
!git clone -q -b claude/pytorch-rewrite-fhuwl4 https://github.com/hanialshater/zestxml.git
%cd /content/zestxml
!pip install -q nltk scikit-learn pytest
!python -c "import nltk; nltk.download('reuters', quiet=True)"

# GZ-NPM rebuilds from the committed snapshot, so numbers match the CPU runs exactly
!python tools/make_npm.py GZXML-Datasets/GZ-NPM
!python tools/make_reuters.py GZXML-Datasets/GZ-Reuters-90
!python -m pytest tests -q | tail -3

In [ ]:
# the GZ-NPM reference run, on GPU this time. ~2 min. Everything else is compared to this.
!python run_torch.py \
  -trn_X_Xf GZXML-Datasets/GZ-NPM/trn_X_Xf.txt -tst_X_Xf GZXML-Datasets/GZ-NPM/tst_X_Xf.txt \
  -Y_Yf GZXML-Datasets/GZ-NPM/Y_Yf.txt -trn_X_Y GZXML-Datasets/GZ-NPM/trn_X_Y.txt \
  -tst_X_Y GZXML-Datasets/GZ-NPM/tst_X_Y.txt -Xf GZXML-Datasets/GZ-NPM/Xf.txt \
  -Yf GZXML-Datasets/GZ-NPM/Yf.txt -res_dir Results/npm -model_dir Results/npm/model \
  -type all -device cuda -num_thread 0 \
  -bs_count 40 -bs_alpha 0.02 -bs_direct_wt 0.8 -shortyK 100 \
  -bilinear_classifier_cost 5 -bilinear_normalize 0 2>&1 | grep -E 'STAT|acc :|finished|epoch 20'
!python tools/eval_xc.py Results/npm/score_mat.bin GZXML-Datasets/GZ-NPM

**Expected (CPU reference):** P@1 73.04, P@5 40.69, PSP@1 19.10, PSP@5 28.62, unseen P@1
52.11, shortlist recall 71.8%. The metrics should reproduce within noise; the wall time
is the thing to watch, since on CPU this took ~52 s against the C++'s 23 s.

## Stage 1 — GPU benchmark on a real dataset (GZ-Eurlex-4.3K)

Downloads the paper's dataset (~1 GB from Drive), builds the C++ reference, runs both
implementations with identical hyper-parameters, reports metrics, wall time and peak GPU
memory. This is the claim that has never been tested: the port exists for GPU throughput,
and on CPU it is 2–4× *slower* than the C++.

If it runs out of memory, lower `-max_elems` / `-dense_elems` / `-batch_size` in
`tools/colab_benchmark.sh` — every stage is chunked, so OOM is a knob, not a wall.

In [ ]:
!bash tools/colab_benchmark.sh GZ-Eurlex-4.3K both

## Stage 2 — SPLADE-style learned sparse, with the leak fixed

The CPU sweep's SPLADE-without-BERT collapsed on zero-shot: unseen P@1 fell **36.79 → 23.00**
while seen rose 67.71 → 72.78. Diagnosis: every label carries a unique `__label__i__name`
feature, so the learned label expansion turns it into a free per-label vector and gradient
descent memorises into it instead of learning the shared token map.

`-drop_label_id 1` removes that route; `-norm_labels 1` kills the norm/popularity prior.
Four configurations, so the fix is attributable rather than a single number.

In [ ]:
for name, flags in [
    ('splade-base',      '-drop_label_id 0 -norm_labels 0'),
    ('splade-nodid',     '-drop_label_id 1 -norm_labels 0'),
    ('splade-norm',      '-drop_label_id 0 -norm_labels 1'),
    ('splade-both',      '-drop_label_id 1 -norm_labels 1'),
]:
    print('=' * 30, name)
    !python baselines/splade.py -data GZXML-Datasets/GZ-NPM -res Results/{name} -epochs 6 {flags} 2>&1 | tail -3
    !python tools/eval_xc.py Results/{name}/score_mat.bin GZXML-Datasets/GZ-NPM

## Stage 3 — does a pretrained encoder beat the candidate ceiling?

**The decisive measurement.** Shortlist recall@100 was pinned at 71.8% under every
configuration tried on CPU — exact map, fastText, Numberbatch, low-rank, graph
propagation. Every re-ranking idea is fighting over that fixed ceiling, so the only way
forward is better candidate generation.

With static word vectors, dense retrieval recalled 39.3% — far *worse* than lexical. The
open question is whether that was the embeddings or the approach. A real encoder answers it.

Try a small model first; `intfloat/e5-base-v2` or `BAAI/bge-base-en-v1.5` are the obvious
upgrades if MiniLM shows signal.

In [ ]:
!pip install -q sentence-transformers
!python tools/hf_dense_probe.py --data GZXML-Datasets/GZ-NPM \
    --lexical Results/npm/shortlist.bin --out Results/npm-hybrid \
    --model sentence-transformers/all-MiniLM-L6-v2 --k 100

Read the three rows against each other at equal budget. The numbers to beat, measured on
CPU with static vectors: lexical 71.8 all / 54.0 unseen; dense 39.3; hybrid 57.3 unseen
(+3.3 over lexical) at the same 100 candidates. If the encoder's dense row clears lexical
on the unseen split, candidate generation stops being the bottleneck and stage 4 matters.

## Stage 4 — feed the hybrid shortlist through the full pipeline

Recall is necessary, not sufficient: extra candidates also give the ranker more ways to be
wrong. On CPU, injecting semantic links into *scoring* helped recall and hurt unseen
precision, so this needs measuring, not assuming.

In [ ]:
# stage 1 of the model is unchanged; reuse it and only redo predict with the new candidates
!cp -r Results/npm/model Results/npm-hybrid/model 2>/dev/null || true
!python run_torch.py \
  -trn_X_Xf GZXML-Datasets/GZ-NPM/trn_X_Xf.txt -tst_X_Xf GZXML-Datasets/GZ-NPM/tst_X_Xf.txt \
  -Y_Yf GZXML-Datasets/GZ-NPM/Y_Yf.txt -trn_X_Y GZXML-Datasets/GZ-NPM/trn_X_Y.txt \
  -tst_X_Y GZXML-Datasets/GZ-NPM/tst_X_Y.txt -Xf GZXML-Datasets/GZ-NPM/Xf.txt \
  -Yf GZXML-Datasets/GZ-NPM/Yf.txt -res_dir Results/npm-hybrid -model_dir Results/npm-hybrid/model \
  -type predict -device cuda -num_thread 0 -bilinear_normalize 0 \
  -shortlist_file Results/npm-hybrid/hybrid_shortlist.bin 2>&1 | grep -E 'STAT|finished'
!python tools/eval_xc.py Results/npm-hybrid/score_mat.bin GZXML-Datasets/GZ-NPM

## Stage 5 — Renee (Microsoft, MLSys 2023)

**This stage is self-contained: run it on a fresh runtime without any earlier stage.**
It clones both repos, rebuilds the dataset and prepares Renee. Every cell is safe to
re-run.

An end-to-end trained one-vs-all XMC model with a transformer encoder — a current method,
and the closest thing to a state-of-the-art reference runnable here.

**Read this before interpreting the result.** Renee learns one classifier vector per
label, so a label with no training positive has an *untrained* vector: expect ~0 on the
unseen slice, exactly like a one-vs-all baseline. That is not a defect, it is the trade —
and it is why label-text augmentation (`CreateAugData.py`, which turns each label's own
text into a training point) is the configuration worth running for a generalized
zero-shot comparison.

Deviations from their README, and why: conda is skipped (Colab already has torch+CUDA);
apex is replaced by a torch shim (it is not on PyPI, `pip install apex` fetches an
unrelated Pyramid library, and building it takes 20 minutes — Renee uses it for two fused
optimizers only); `--use-ngame-encoder` is omitted since those weights are a separate
download. So these numbers will sit **below** Renee's published ones. Treat this as "a
current method, honestly configured, on our data", not a reproduction of the paper.

Compare against the ZestXML reference on the same dataset: **P@1 73.04, P@5 40.69,
PSP@1 19.10, PSP@5 28.62, unseen-only P@1 52.11**.

In [ ]:
%%bash
set -e
cd /content
# our repo: clone, or refresh if it is already here
if [ -d zestxml/.git ]; then git -C zestxml pull -q; else
  git clone -q -b claude/pytorch-rewrite-fhuwl4 https://github.com/hanialshater/zestxml.git
fi
[ -d renee/.git ] || git clone -q https://github.com/microsoft/renee.git
echo "repos ready:" && ls -d /content/zestxml /content/renee

In [ ]:
!pip install -q scikit-learn transformers sentence-transformers cython seaborn
!pip install -q git+https://github.com/kunaldahiya/pyxclib.git
# NVIDIA apex is not on PyPI: `pip install apex` installs an unrelated Pyramid auth
# library whose import breaks dl_base.py. Remove it if a previous attempt installed it.
!pip uninstall -y -q apex 2>/dev/null; true

In [ ]:
%%bash
set -e
cd /content/zestxml
python tools/make_npm.py GZXML-Datasets/GZ-NPM        # rebuilds from the committed snapshot

cd /content/renee
cp /content/zestxml/colab/apex.py apex.py             # torch stand-in for the fused optimizers
# transformers >= 4.5x removed BertTokenizer.batch_encode_plus; __call__ takes the same kwargs
sed -i 's/tokenizer\.batch_encode_plus(/tokenizer(/' utils/CreateTokenizedFiles.py
mkdir -p xc/Datasets
rm -rf xc/Datasets/GZ-NPM xc/Datasets/GZ-NPM-Aug      # start clean, previous runs leave partials
cp -r /content/zestxml/GZXML-Datasets/GZ-NPM xc/Datasets/GZ-NPM
python -c "import apex; print('apex shim ok:', apex.optimizers.FusedAdam)"
echo "--- dataset ---" && wc -l < xc/Datasets/GZ-NPM/trn_X.txt && head -1 xc/Datasets/GZ-NPM/trn_X_Y.txt

In [ ]:
%cd /content/renee
!python -W ignore -u utils/CreateTokenizedFiles.py \
  --data-dir xc/Datasets/GZ-NPM --max-length 32 \
  --tokenizer-type bert-base-uncased --tokenize-label-texts

# label-text augmentation: gives every label, including unseen ones, a training signal
!python utils/CreateAugData.py --data-dir xc/Datasets/GZ-NPM \
  --tokenization-folder bert-base-uncased-32 --max-len 32

# check the tokenised rows line up before spending an hour of GPU on them: Renee maps
# line N of trn_X.txt to row N of trn_X_Y.txt and never checks the lengths itself
import os
d = 'xc/Datasets/GZ-NPM-Aug/bert-base-uncased-32'
for f in sorted(os.listdir(d)):
    print(f"{f:34s} {os.path.getsize(f'{d}/{f}') // (8 * 32):>7d} rows")
print('expect trn_doc 28350 (25127 points + 3223 label texts), tst_doc 8376, lbl 3223')

In [ ]:
# ~45-90 min on an A100. Drop --epochs to 20 for a cheaper first look.
!cd /content/renee && python main.py --epochs 50 --batch-size 32 --lr1 0.05 --lr2 1e-5 \
  --warmup 1000 --data-dir xc/Datasets/GZ-NPM-Aug --maxlen 32 \
  --tf sentence-transformers/msmarco-distilbert-base-v4 \
  --dropout 0.85 --pre-tok --wd1 1e-4 --noloss --fp16xfc --expname gznpm-aug

In [ ]:
# Renee's output layout is version dependent: find its score matrix, then convert it
# so the same evaluator scores both systems.
!find /content/renee -name '*.npz' -newermt '-3 hours' 2>/dev/null | head
!ls -R /content/renee/Results 2>/dev/null | head -30

In [ ]:
SCORES = ''  # paste the .npz path printed above

if SCORES:
    import sys, scipy.sparse as sp, torch
    sys.path.insert(0, '/content/zestxml')
    from zestxml.csr import CSR
    from zestxml.io import write_bin_smat, ensure_dir
    m = sp.load_npz(SCORES).tocsr()
    print('renee scores', m.shape, m.nnz)
    ensure_dir('/content/zestxml/Results/renee')
    write_bin_smat(CSR(torch.as_tensor(m.indptr).long(), torch.as_tensor(m.indices).long(),
                       torch.as_tensor(m.data).float(), m.shape),
                   '/content/zestxml/Results/renee/score_mat.bin')
    !cd /content/zestxml && python tools/eval_xc.py Results/renee/score_mat.bin GZXML-Datasets/GZ-NPM
else:
    print('set SCORES to the path printed above')

## Stage 6 — one table

In [ ]:
%cd /content/zestxml
import glob, subprocess, re

rows = []
for path in sorted(glob.glob('Results/*/score_mat.bin')):
    name = path.split('/')[1]
    out = subprocess.run(['python', 'tools/eval_xc.py', path, 'GZXML-Datasets/GZ-NPM'],
                         capture_output=True, text=True).stdout
    got = {}
    for line in out.splitlines():
        m = re.match(r'(all labels|unseen only|seen only)\s+\d+\s+(.*)', line.strip())
        if m:
            got[m.group(1)] = [float(x) for x in m.group(2).split()]
    if 'all labels' in got:
        a, u = got['all labels'], got.get('unseen only', [float('nan')] * 7)
        rows.append((name, a[0], a[2], a[4], a[6], u[0]))

rows.sort(key=lambda r: -r[5])
print(f"{'method':<28}{'P@1':>8}{'P@5':>8}{'PSP@1':>8}{'PSP@5':>8}{'unseen P@1':>12}")
for r in rows:
    print(f'{r[0]:<28}{r[1]:8.2f}{r[2]:8.2f}{r[3]:8.2f}{r[4]:8.2f}{r[5]:12.2f}')

### What to send back

The stage 6 table, plus the `[STAT]` lines from stage 1 (wall time and peak GPU memory)
and the three-row recall table from stage 3. Those three things settle, in order: whether
the GPU port is worth having, whether the candidate ceiling can be broken, and where the
port stands against a current method.

Caveat that applies to the whole notebook: GZ-NPM and GZ-Reuters-90 are self-built
datasets, so none of these numbers are comparable to published XMC results. Only stage 1
uses a dataset from the paper.